# 一、前言

通过对四个材料的解读，对反向传播和梯度更新等概念及公式进行理解和推演，再用程序测试验证，通过自测题的检测

# 二、四段材料笔记

## 材料一：3B1B《Backpropagation calculus》

1.1 他用的网络和记号<br>
单神经元网络起步：每层 1 个神经元，3 个权重 + 3 个偏置（输入层无偏置）
$z^{(L)} = w^{(L)} a^{(L-1)} + b^{(L)}$，$a^{(L)} = \sigma(z^{(L)})$，单样本代价 $C_0 = (a^{(L)} - y)^2$
上标表示层，下标表示神经元（$w^{(L)}_{jk}$：连接第 $k$ 个神经元到第 $j$ 个神经元的边）

1.2 链式法则的三个"零件"（必须能默写）<br>
$\frac{\partial z^{(L)}}{\partial w^{(L)}} = a^{(L-1)}, \qquad \frac{\partial a^{(L)}}{\partial z^{(L)}} = \sigma'(z^{(L)}), \qquad \frac{\partial C_0}{\partial a^{(L)}} = 2(a^{(L)} - y)$
合起来：<br>
$\frac{\partial C_0}{\partial w^{(L)}} = a^{(L-1)} \cdot \sigma'(z^{(L)}) \cdot 2(a^{(L)} - y)$

1.3 三个"零件"的直觉（易漏，但这是 3B1B 的精髓）<br>
$\partial z/\partial w = a^{(L-1)}$："一起放电的神经元会连在一起"（fire together, wire together）——前一层激活越高，这个权重对代价的影响越大<br>
$\partial a/\partial z = \sigma'(z)$：加权和的斜率；斜率平缓时，权重怎么改都不太影响激活<br>
$\partial C_0/\partial a = 2(a-y)$：输出偏差越大，梯度越猛——"错得越离谱，改得越用力"

1.4 多神经元的关键升级（路径求和！）<br>
前一层第 $k$ 个神经元通过所有出边影响下一层所有 $j$，所以：<br>
$\frac{\partial C_0}{\partial a_k^{(L-1)}} = \sum_{j} \frac{\partial z_j^{(L)}}{\partial a_k^{(L-1)}} \cdot \frac{\partial a_j^{(L)}}{\partial z_j^{(L)}} \cdot \frac{\partial C_0}{\partial a_j^{(L)}}$
这就是"误差沿多条路径分摊，路径之间相加"的数学形式。

1.5 其余必记点<br>
偏置的梯度更简单：$\partial z^{(L)}/\partial b^{(L)} = 1$（链上少一项）<br>
全数据集：$C = \frac{1}{n}\sum_k C_k$，梯度是各样本梯度的平均<br>
核心思想：每个偏导数都是"敏感性"——微小扰动被逐层放大的比率

## 材料二：李宏毅《反向传播》

2.1 拆解<br>
$\frac{\partial l}{\partial w} = \underbrace{\frac{\partial z}{\partial w}}_{\text{Forward Pass 算}} \times \underbrace{\frac{\partial l}{\partial z}}_{\text{Backward Pass 算}}$

Forward Pass 算 $\partial z/\partial w$：因为 $z = wx + b$，$\partial z/\partial w = x$ = 这条边的输入激活值，前向计算时顺手就有
Backward Pass 算 $\partial l/\partial z$：这是难点，需要从输出层往回递归

2.2 Backward Pass 的递归式（核心公式）<br>
$\frac{\partial l}{\partial z_i} = \sum_j \frac{\partial l}{\partial z'_j} \cdot \frac{\partial z'_j}{\partial z_i}, \qquad \frac{\partial z'_j}{\partial z_i} = w_{ij}\,\sigma'(z_i)$
代入后：<br>
$\frac{\partial l}{\partial z_i} = \sum_j \frac{\partial l}{\partial z'_j} \cdot w_{ij} \cdot \sigma'(z_i)$

递归基：输出层的 $\partial l/\partial z$ 由损失函数直接给出
易错点：$\partial z'_j/\partial z_i$ 是把 $z_i$ 当自变量、对 $z'_j = w_{ij}\sigma(z_i)$ 求导 → 得到 $w_{ij}\sigma'(z_i)$，别丢掉 $\sigma'(z_i)$

2.3 两个概念性结论<br>
"反向传播就是另一个神经网络"：计算图结构相同、方向相反；正向节点做"线性组合+激活"，反向节点做"误差加权和×斜率"<br>
复杂度对比：数值微分 = 每个参数跑一次前向（$O(参数个数)$ 次）；反向传播 = 一次前向 + 一次反向（$O(1)$ 个 pass）

## 材料三：d2l 4.7《前向传播、反向传播和计算图》

3.1 记号<br>
$\mathbf{x}\in\mathbb{R}^d$，$\mathbf{W}^{(1)}\in\mathbb{R}^{h\times d}$，$\mathbf{W}^{(2)}\in\mathbb{R}^{q\times h}$；中间变量：$\mathbf{z}$、$\mathbf{h}$（既是激活又是隐藏层维度名，注意区分）、$\mathbf{o}$。

3.2 前向六式（(4.7.1)–(4.7.6)）<br>
$\mathbf{z}=\mathbf{W}^{(1)}\mathbf{x} \quad (4.7.1) \qquad \mathbf{h}=\phi(\mathbf{z}) \quad (4.7.2) \qquad \mathbf{o}=\mathbf{W}^{(2)}\mathbf{h} \quad (4.7.3)$
$L=l(\mathbf{o},y) \quad (4.7.4) \qquad s=\frac{\lambda}{2}\left(\|\mathbf{W}^{(1)}\|_F^2+\|\mathbf{W}^{(2)}\|_F^2\right) \quad (4.7.5 L2 正则化表达式) \qquad J=L+s \quad (4.7.6)$

正则项 $s$ 是两个权重矩阵的 Frobenius 范数平方<br>
目标函数 $J$ = 损失 + 正则，所有梯度都是对 $J$ 求，不是对 $L$

3.3 prod 运算符（(4.7.7)，本节最容易卡的概念）<br>
$\frac{\partial \mathsf{Z}}{\partial \mathsf{X}} = \text{prod}\left(\frac{\partial \mathsf{Z}}{\partial \mathsf{Y}}, \frac{\partial \mathsf{Y}}{\partial \mathsf{X}}\right)$

它不是在定义一个新的数学运算，而是说"这里要执行合适的梯度乘法（含转置、换序、reshape）
每个 prod 展开后是什么形状，重点

3.4 反向推导五步（(4.7.8)–(4.7.14)，主战场）<br>
| 步骤 | 公式 | 含义 |
|---|---|---|
| 1 | (4.7.8) $\partial J/\partial L = 1,\ \partial J/\partial s = 1$ | 根节点分流 |
| 2 | (4.7.9) $\frac{\partial J}{\partial \mathbf{o}} = \frac{\partial L}{\partial \mathbf{o}} \in \mathbb{R}^q$ | 正则项与 $\mathbf{o}$ 无关 |
| 3 | (4.7.10) $\frac{\partial s}{\partial \mathbf{W}^{(1)}} = \lambda\mathbf{W}^{(1)},\ \frac{\partial s}{\partial \mathbf{W}^{(2)}} = \lambda\mathbf{W}^{(2)}$ | **Frobenius 范数的梯度 = 自己乘 λ** |
| 4 | (4.7.11) $\frac{\partial J}{\partial \mathbf{W}^{(2)}} = \frac{\partial J}{\partial \mathbf{o}}\mathbf{h}^\top + \lambda\mathbf{W}^{(2)}$ | 输出层权重：数据项 + 正则项 |
| 5 | (4.7.12) $\frac{\partial J}{\partial \mathbf{h}} = {\mathbf{W}^{(2)}}^\top\frac{\partial J}{\partial \mathbf{o}}$ | **穿层回传的关键** |
| 6 | (4.7.13) $\frac{\partial J}{\partial \mathbf{z}} = \frac{\partial J}{\partial \mathbf{h}} \odot \phi'(\mathbf{z})$ | 逐元素乘（激活） |
| 7 | (4.7.14) $\frac{\partial J}{\partial \mathbf{W}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}}\mathbf{x}^\top + \lambda\mathbf{W}^{(1)}$ | 输入层权重 |

3.5 每个 prod 的形状核对（重点）<br>
(4.7.11)：$\partial J/\partial \mathbf{o}\in\mathbb{R}^q$ 与 $\mathbf{h}^\top\in\mathbb{R}^{h}$ 相乘 → $\mathbb{R}^{q\times h}$ ✓（匹配 $\mathbf{W}^{(2)}$）<br>
(4.7.12)：$\mathbf{W}^{(2)\top}\in\mathbb{R}^{h\times q}$ 乘 $\partial J/\partial \mathbf{o}\in\mathbb{R}^{q}$ → $\mathbb{R}^{h}$ ✓（匹配 $\mathbf{h}$）<br>
(4.7.13)：两个 $\mathbb{R}^h$ 逐元素 → $\mathbb{R}^h$ ✓<br>
(4.7.14)：$\partial J/\partial \mathbf{z}\in\mathbb{R}^{h}$ 与 $\mathbf{x}^\top\in\mathbb{R}^{d}$ 相乘 → $\mathbb{R}^{h\times d}$ ✓（匹配 $\mathbf{W}^{(1)}$）<br>
转置出现在哪，为什么出现在那，能自己讲出来 = 真懂 prod。

3.6 训练 vs 预测（4.7.4 节）<br>
前向存下的 $\mathbf{z},\mathbf{h},\mathbf{o}$ 反向要复用 → 训练内存 ≈ 预测 2 倍，且中间值大小 ∝ 层数 × batch 大小 → 大 batch 深网络容易 OOM<br>
前向和反向相互依赖、交替进行（前向需要参数当前值，反向需要前向的中间值）

3.7 练习 2（推荐做）：加偏置<br>
推导带 $\mathbf{b}^{(1)},\mathbf{b}^{(2)}$ 的正反向方程：$\partial J/\partial \mathbf{b}^{(2)}=\partial J/\partial \mathbf{o}$，$\partial J/\partial \mathbf{b}^{(1)}=\partial J/\partial \mathbf{z}$（偏置的梯度 = 该层未乘权重的梯度量）

## 材料四：d2l 附录 22.4《Multivariable Calculus》

4.1 22.4.1–22.4.3（梯度几何）<br>
(22.4.5) 一阶近似：$L(\mathbf{w}+\boldsymbol{\epsilon}) \approx L(\mathbf{w}) + \boldsymbol{\epsilon}\cdot\nabla_{\mathbf{w}}L(\mathbf{w})$
(22.4.10) 最陡下降方向 = $-\nabla L$（让 $\cos\theta=-1$，即与梯度方向相反）
极值必要条件：$\nabla f=0$（临界点），但梯度为零不一定是极值

4.2 22.4.4 多元链式法则 = 路径求和（本材料最重要的概念）<br>
例子：(22.4.14) $f(u,v)=(u+v)^2$，$u(a,b)=(a+b)^2$，$v(a,b)=(a-b)^2$，$a(w,x,y,z)=(w+x+y+z)^2$，$b(w,x,y,z)=(w+x-y-z)^2$

(22.4.16) 直接展开 $\partial f/\partial w$：灾难级长式，且 $\partial f/\partial x$ 与它有大量重复共享项<br>
(22.4.18) 正确的拆法（两条路径）：<br>
$\frac{\partial f}{\partial a} = \underbrace{\frac{\partial f}{\partial u}\frac{\partial u}{\partial a}}_{\text{路径 a→u→f}} + \underbrace{\frac{\partial f}{\partial v}\frac{\partial v}{\partial a}}_{\text{路径 a→v→f}}$
(22.4.19) 图 22.4.2 中 $\partial f/\partial y$ = 三条路径求和<br>
一句话：一元链式法则是"乘积"，多元链式法则是"沿所有路径的乘积之和"。

4.3 22.4.5 反向传播算法（两个代码块的对比是全节精华）<br>
同一个函数 $f$，两种算 $\partial f/\partial w$ 的方式：<br>
代码块 1（正向展开）代码块 2（反向分解）链式法则怎么用$\partial f/\partial w = \partial f/\partial u\cdot\partial u/\partial w + \dots$，逐层向下展开 $\partial w$ 分母先算 $\partial f/\partial u, \partial f/\partial v$，再算 $\partial f/\partial a, \partial f/\partial b$，最后 $\partial f/\partial w$方向从输入往输出从输出（$f$）往输入代价每个变量重来一遍，重复共享项中间梯度只算一次，全部复用数值例w=-1,x=0,y=-2,z=1同上<br>
数值验证（已核对）：$w=-1,x=0,y=-2,z=1$ 时 $f=1024$，$\partial f/\partial w=\partial f/\partial x=\partial f/\partial y=\partial f/\partial z=-4096$

得名原因（原文）："从 $f$ 往输入方向算梯度，而不是从输入往 $f$ 算，这就是它叫 backpropagation 的原因"<br>
两阶段：forward pass（算函数值 + 单步偏导）→ backward pass（从后往前组合出所有梯度）<br>
最后 f.backward() 一行完成全部——这就是 PyTorch 里 .backward() 的底细

4.4 22.4.7 矩阵微积分（可选，快速了解）<br>
denominator layout：导数排成分母的形状<br>
转置"凭空出现"的原因：要匹配分母形状（如 (22.4.52) $-2\mathbf{U}^\top(\mathbf{X}-\mathbf{U}\mathbf{V})$）<br>
实用技巧：先写 1×1 标量版本的导数，再猜矩阵版本（加转置配形状）

# 三、代码验证

## 1.数值梯度工具（后面都要用）

In [10]:
import torch
def numeric_grad(fn, params, h = 1e-4):
    """中心差分数值梯度。
    fn: 返回标量损失（内部用 params 当前值重算）
    params: requires_grad 的张量列表
    """
    grads = []   # 用来保存最后算出来所有参数的数值梯度
    for p in params:   # 循环每一个需要求梯度的张量（神经网络的权重、偏置）
        g = torch.zeros_like(p)  # 创建一个和参数 p 形状一模一样、初始全 0 的张量，用来存这个参数的梯度
        flat = p.detach().flatten()  # 关键：detach 后再取视图，可安全原地修改
        for k in range(flat.numel()):  # 这个参数一共有多少个元素。循环遍历每一个权重元素，挨个求偏导
            old = flat[k].item()  # 记下当前这个权重原始的值，后面算完要恢复回去
            flat[k] = old + h; lp = fn()  # 把第 k 个元素加上微小偏移 h，调用损失函数 fn()，得到损失值 f(x+h)，存到 lp
            flat[k] = old - h; lm = fn()  # 把第 k 个元素减去微小偏移 h，调用损失函数 fn()，得到损失值 f(x-h)，存到 lm
            flat[k] = old  # 复原权重的值,如果不改回去，下一轮循环参数就已经变了，梯度全部算错
            g.flatten()[k] = (lp - lm)/(2 * h)  # 套用中心差分公式，算出这个元素的偏导数，写入梯度数组对应位置
        grads.append(g)  # 把当前参数算完的梯度，加入梯度列表。最后返回全部参数的数值梯度
    return grads

核心数学原理（中心差分）对参数 \(x_k\)，导数近似公式：<br>
$\frac{\partial f}{\partial x_k}\approx \frac{f(x+h)-f(x-h)}{2h}$<br>
相比单边差分 $\frac{f(x+h)-f(x)}{h}$，中心差分精度更高
h 取很小的数，典型值：1e‑4

放到梯度校验的场景，它的作用<br>
PyTorch 的 loss.backward() 是解析梯度（精确导数）<br>
numeric_grad 的中心差分得到数值梯度（近似导数）<br>
对比这两份梯度，看是不是几乎相等<br>
差不多 → 手动反向传播代码没问题<br>
差很多 → 反向传播写错了，有 bug

## 2.3B1B 单神经元：三个零件相乘

In [11]:
# ============ Cell 1 · 3B1B 单神经元链式法则 ============
# 验证：∂C/∂w = a^(L-1) · σ'(z) · 2(a-y)（三个零件）
# 三方对比：手推 / autograd / 数值梯度
x = torch.tensor(0.5); y = torch.tensor(0.8)

w1 = torch.tensor(1.2, requires_grad=True)
w2 = torch.tensor(-0.7, requires_grad=True)
w3 = torch.tensor(0.9, requires_grad=True)
b1 = torch.tensor(0.1, requires_grad=True)
b2 = torch.tensor(-0.3, requires_grad=True)
b3 = torch.tensor(0.2, requires_grad=True)

def sigma(z): 
    return torch.sigmoid(z)

def fn():   # 供数值梯度用：当前参数下的损失
    a1 = sigma(w1 * x + b1)
    a2 = sigma(w2 * a1 + b2)
    a3 = sigma(w3 * a2 + b3)
    return (a3 - y) ** 2  # 损失（平方误差）

# --- forward + autograd ---
z1 = w1 * x + b1; a1 = sigma(z1)
z2 = w2 * a1 + b2; a2 = sigma(z2)
z3 = w3 * a2 + b3; a3 = sigma(z3)
C0 = (a3 - y) ** 2
C0.backward()

# --- 3B1B 手推：三个零件逐层往回 ---
with torch.no_grad():
    dC_da3 = 2 * (a3 - y)                                     # 零件③：输出偏差
    dC_dw3 = a2 * sigma(z3) * (1 - sigma(z3)) * dC_da3        # a^(L-1)·σ'(z)·零件③
    dC_da2 = w3 * sigma(z3) * (1 - sigma(z3)) * dC_da3        # 误差往回分摊
    dC_dw2 = a1 * sigma(z2) * (1 - sigma(z2)) * dC_da2
    dC_da1 = w2 * sigma(z2) * (1 - sigma(z2)) * dC_da2
    dC_dw1 = x  * sigma(z1) * (1 - sigma(z1)) * dC_da1
    dC_db3 = sigma(z3) * (1 - sigma(z3)) * dC_da3
    dC_db2 = sigma(z2) * (1 - sigma(z2)) * dC_da2
    dC_db1 = sigma(z1) * (1 - sigma(z1)) * dC_da1

names = ["w1", "w2", "w3", "b1", "b2", "b3"]
manual = [dC_dw1, dC_dw2, dC_dw3, dC_db1, dC_db2, dC_db3]
auto   = [w1.grad, w2.grad, w3.grad, b1.grad, b2.grad, b3.grad]
num    = numeric_grad(fn, [w1, w2, w3, b1, b2, b3])

for n, m, a, ng_ in zip(names, manual, auto, num):
    print(f"{n}: 手推={m.item():+.6f} autograd={a.item():+.6f} "
          f"|手推-auto|={abs(m.item()-a.item()):.2e} |数值-auto|={abs(ng_.item()-a.item()):.2e}")

w1: 手推=+0.001291 autograd=+0.001291 |手推-auto|=2.33e-10 |数值-auto|=5.71e-06
w2: 手推=-0.011116 autograd=-0.011116 |手推-auto|=9.31e-10 |数值-auto|=4.08e-06
w3: 手推=-0.027062 autograd=-0.027062 |手推-auto|=1.86e-09 |数值-auto|=2.06e-05
b1: 手推=+0.002582 autograd=+0.002582 |手推-auto|=4.66e-10 |数值-auto|=7.20e-06
b2: 手推=-0.016636 autograd=-0.016636 |手推-auto|=1.86e-09 |数值-auto|=2.11e-05
b3: 手推=-0.085378 autograd=-0.085378 |手推-auto|=7.45e-09 |数值-auto|=1.62e-04


网络结构：
$x\to z_1=w_1x+b_1\to a_1=\sigma(z_1)\to z_2=w_2a_1+b_2\to a_2=\sigma(z_2)\to z_3=w_3a_2+b_3\to a_3=\sigma(z_3)$

反向传播链条（3B1B 链式拆解）<br>
1.输出层误差项<br>
$\frac{\partial C}{\partial a_3}=2(a_3-y)$<br>
2. Sigmoid 导数：<br>
$\sigma'(z)=\sigma(z)(1-\sigma(z)$<br>
3. 权重梯度通式：<br>
$\frac{\partial C}{\partial w_L}=a^{L-1}\cdot\sigma'(z_L)\cdot\frac{\partial C}{\partial a_L}$<br>
4. 偏置梯度通式：<br>
$\frac{\partial C}{\partial b_L}=\sigma'(z_L)\cdot\frac{\partial C}{\partial a_L}$<br>
5. 误差向前一层传递：<br>
$\frac{\partial C}{\partial a_{L-1}}=w_L\cdot\sigma'(z_L)\cdot\frac{\partial C}{\partial a_L}$


梯度校验结果<br>
1.三组梯度含义<br>
手推梯度：基于链式法则手动推导解析解（理论真值）<br>
autograd 梯度：PyTorch 自动微分求出的解析梯度<br>
数值梯度：中心差分法得到的梯度近似值，用于校验<br>

2.误差结果一览<br>
|参数|手推‑自动梯度绝对误差|数值‑自动梯度绝对误差|
|---|---|---|
|w1|2.33e‑10|5.71e‑6<br>|
|w2|9.31e‑10|4.08e‑6<br>|
|w3|1.86e‑9|2.06e‑5<br>|
|b1|4.66e‑10|7.20e‑6<br>|
|b2|1.86e‑9|2.11e‑5<br>|
|b3|7.45e‑9|1.62e‑4<br>|

3.结果判定<br>
1) 手推梯度 VS Autograd 自动梯度全部误差量级集中在 $10^{-9}\sim10^{-10}$<br>
误差仅来自计算机浮点数精度，无理论偏差。<br>
结论：你手写的反向传播链式法则公式完全正确。<br>
1) 数值梯度 VS Autograd 自动梯度最大绝对误差：$\boldsymbol{1.62\times 10^{-4}}$，远小于梯度检验警戒线 $10^{-3}$；<br>
b3 误差最大是因为它位于输出层，梯度本身数值更大，属于正常现象，其相对误差仅约 0.19%。<br>
结论：数值梯度校验通过，前向损失函数实现没有问题。

## 3.3B1B 多神经元：误差沿出边加权回传

In [14]:
# ============ 2-2-1 网络，δ 递推 ============
# 验证：δ^l_k = σ'(z^l_k) · Σ_j w^l_jk δ^(l+1)_j（对 j 求和，路径分摊）
#       且 ∂C/∂w^l_jk = a^(l-1)_k · δ^l_j（外层积）
torch.manual_seed(42)
x = torch.randn(2); y = torch.randn(1)
W1 = torch.randn(2, 2, requires_grad=True); b1 = torch.randn(2, requires_grad=True)
W2 = torch.randn(1, 2, requires_grad=True);  b2 = torch.randn(1, requires_grad=True)
# --- forward + autograd ---
z1 = W1 @ x + b1; a1 = torch.sigmoid(z1)
z2 = W2 @ a1 + b2; a2 = torch.sigmoid(z2)
C = ((a2 - y) ** 2).sum()
C.backward()
# --- 手写 δ 递推（3B1B 误差分摊，矩阵形式）---
with torch.no_grad():
    dC_da2 = 2 * (a2 - y)                                  # 输出偏差
    delta2 = dC_da2 * torch.sigmoid(z2) * (1 - torch.sigmoid(z2))   # δ^2
    dC_dW2 = delta2.unsqueeze(1) * a1.unsqueeze(0)         # (1,1)×(1,2)→(1,2) 外层积
    delta1 = (W2.T @ delta2) * torch.sigmoid(z1) * (1 - torch.sigmoid(z1))  # δ^1 = (W2ᵀδ2)⊙σ'(z1)
    dC_dW1 = delta1.unsqueeze(1) * x.unsqueeze(0)          # (2,1)×(1,2)→(2,2) 外层积
print("W2 梯度差:", (W2.grad - dC_dW2).abs().max().item())
print("W1 梯度差:", (W1.grad - dC_dW1).abs().max().item())
print("b2 梯度差:", (b2.grad - delta2).abs().max().item())
print("b1 梯度差:", (b1.grad - delta1).abs().max().item())

W2 梯度差: 0.0
W1 梯度差: 0.0
b2 梯度差: 0.0
b1 梯度差: 0.0


torch.unsqueeze()<br>
在指定位置，给张量新增一个长度为 1 的维度。不改变数据，只改变形状。<br>
torch.unsqueeze(input, dim)<br>
或者张量方法版<br>
tensor.unsqueeze(dim)<br>
dim：你想在哪一个索引位置插入新维度（从 0 开始）<br>
在这儿此函数将一维张量升为二维；[] -> [[ ]],用以方便后续的广播
.abs () —— 取绝对值

误差递推方程 + 权重梯度方程<br>
网络结构：2‑2‑1<br>
输入 2 维 → 隐藏层 2 个神经元 → 输出层 1 个神经元，Sigmoid + MSE 损失

前向传播<br>
$\boldsymbol z^1 = W_1 \boldsymbol x+\boldsymbol b_1,\quad
\boldsymbol a^1=\sigma(\boldsymbol z^1)z^2 = W_2 \boldsymbol a^1+b_2,\quad
a^2=\sigma(z^2)$<br>
损失：<br>
$C=(a^2-y)^2$<br>

输出层误差 $\delta^2$<br>
dC_da2 = 2 * (a2 - y)<br>
delta2 = dC_da2 * torch.sigmoid(z2) * (1 - torch.sigmoid(z2))<br>
公式：<br>
$\boldsymbol\delta^{L}=\frac{\partial C}{\partial \boldsymbol z^{L}}
=\frac{\partial C}{\partial \boldsymbol a^{L}}\odot\sigma'(\boldsymbol z^{L})$<br>
$\odot$：逐元素相乘（哈达玛积）<br>
输出层误差不需要求和，直接由损失导数乘激活导数

上一层误差递推（误差向后分摊）<br>
delta1 = (W2.T @ delta2) * torch.sigmoid(z1) * (1 - torch.sigmoid(z1))br>
对应反向传播误差递推方程 BP‑2：<br>
$\boldsymbol\delta^{l}=\big(W^{l+1}\big)^\mathsf{T}\boldsymbol\delta^{l+1}
\odot\,\sigma'(\boldsymbol z^{l})$<br>
W2.T @ delta2：把后一层的误差，沿着权重反向传播、求和分摊给前一层每一个神经元；<br>
再点乘激活导数，得到本层神经元的误差 $\delta^1$

权重梯度：外积<br>
dC_dW2 = delta2.unsqueeze(1) * a1.unsqueeze(0)br>
BP‑4：<br>
$\frac{\partial C}{\partial W^{l}}=\boldsymbol\delta^{l}\, \big(\boldsymbol a^{l-1}\big)^\mathsf{T}$<br>
delta2 形状 [1] → unsqueeze(1) → (1,1)<br>
a1 形状 [2] → unsqueeze(0) → (1,2)<br>
逐元素相乘等价于矩阵外积，结果：(1, 2)，正好和 W2 的形状匹配<br>
dC_dW1 = delta1.unsqueeze(1) * x.unsqueeze(0)<br>
delta1：[2] → (2,1)；x：[2] → (1,2)<br>
相乘得到 (2, 2)，与 W1 形状完全对应。

偏置梯度<br>
$\frac{\partial C}{\partial \boldsymbol b^{l}}=\boldsymbol\delta^{l}$<br>
$b_2$的梯度 =$\delta^2$<br>
$b_1$ 的梯度 = $\delta^1$

## 4. 李宏毅：手写 backward pass 递归

In [26]:
# ============  手写"反向传播 = 另一个网络" ============
# 验证：不靠 autograd，自己按李宏毅的 Backward Pass 递归式算梯度
# 3->1 网络（3 隐藏神经元），与 autograd 对比
torch.manual_seed(7)
x = torch.randn(2); y = torch.randn(1)
W1 = torch.randn(3, 2, requires_grad=True); b1 = torch.randn(3, requires_grad=True)
W2 = torch.randn(1, 3, requires_grad=True);  b2 = torch.randn(1, requires_grad=True)
def sig(z): return torch.sigmoid(z)
def sigp(z): return sig(z) * (1 - sig(z))
# forward：像框架一样把中间值存下来
z1 = W1 @ x + b1; a1 = sig(z1)
z2 = W2 @ a1 + b2; a2 = sig(z2)
L = ((a2 - y) ** 2).sum()
L.backward()
# 手写 backward pass（从输出层逐层递归）
with torch.no_grad():
    delta2 = 2 * (a2 - y) * sigp(z2)                        # 输出层 δ（递归基）
    dL_dW2 = delta2.unsqueeze(1) * a1.unsqueeze(0)
    dL_db2 = delta2
    delta1 = (W2.T @ delta2) * sigp(z1)                     # 隐藏层 δ（递归一步）
    dL_dW1 = delta1.unsqueeze(1) * x.unsqueeze(0)
    dL_db1 = delta1
for n, m, a in [("W2", dL_dW2, W2.grad), ("b2", dL_db2, b2.grad),
                ("W1", dL_dW1, W1.grad), ("b1", dL_db1, b1.grad)]:
    print(f"{n}: 手写-自动 最大差 = {(m - a).abs().max().item():.2e}")

W2: 手写-自动 最大差 = 0.00e+00
b2: 手写-自动 最大差 = 0.00e+00
W1: 手写-自动 最大差 = 1.46e-11
b1: 手写-自动 最大差 = 1.46e-11


如果 Layer 数更多，delta 的递归式完全一样，只是多循环几层——这就是"另一个网络"：结构相同（一层接一层），方向相反，每层节点做的事 = 误差加权和 × 斜率。现在手写的这 5 行，就是 loss.backward() 内部在 3 层网络里干的事。

## 5.d2l 4.7 矩阵公式（含正则项，非常重要）

In [25]:
# ============ 验证 (4.7.9)–(4.7.14) ============
# 含正则项 λW，激活 tanh；损失 L = ½‖o-y‖²（让 ∂L/∂o = o-y，和 d2l 记号对齐）
torch.manual_seed(0)
d, h_dim, q = 4, 5, 3
x = torch.randn(d); y = torch.randn(q)
W1 = torch.randn(h_dim, d, requires_grad=True)
W2 = torch.randn(q, h_dim, requires_grad=True)
lam = 0.5
def fn():   # 数值梯度用的完整目标 J
    z = W1 @ x
    h = torch.tanh(z)
    o = W2 @ h
    L = 0.5 * ((o - y) ** 2).sum()
    s = lam / 2 * (W1.norm() ** 2 + W2.norm() ** 2)
    return L + s
# --- autograd（对 J 求梯度）---
z = W1 @ x; h = torch.tanh(z); o = W2 @ h          # (4.7.1)(4.7.2)(4.7.3)
L = 0.5 * ((o - y) ** 2).sum()                     # (4.7.4)
s = lam / 2 * (W1.norm() ** 2 + W2.norm() ** 2)    # (4.7.5)
J = L + s                                          # (4.7.6)
J.backward()
# --- 手写 (4.7.9)–(4.7.14) ---
with torch.no_grad():
    dJ_do  = o - y                                              # (4.7.9)
    dJ_dW2 = dJ_do.unsqueeze(1) * h.unsqueeze(0) + lam * W2     # (4.7.11) 数据项+正则项
    dJ_dh  = W2.T @ dJ_do                                       # (4.7.12) 穿层回传
    dJ_dz  = dJ_dh * (1 - h ** 2)                               # (4.7.13) tanh' = 1-tanh²
    dJ_dW1 = dJ_dz.unsqueeze(1) * x.unsqueeze(0) + lam * W1     # (4.7.14)
print("(4.7.11) W2:", (W2.grad - dJ_dW2).abs().max().item())
print("(4.7.14) W1:", (W1.grad - dJ_dW1).abs().max().item())
print("(4.7.12)(4.7.13) 中间量核对:", (W2.grad[:1, :1] - dJ_dW2[:1, :1]).abs().max().item())
# --- 正则项单独验证：∂s/∂W = λW ---
with torch.no_grad():
    print("∂s/∂W1 应等于 λ·W1:", (lam * W1 - (J.grad_fn is not None)) if False else
          f"λW1[0,0]={lam*W1[0,0].item():.4f}")

(4.7.11) W2: 0.0
(4.7.14) W1: 2.9802322387695312e-08
(4.7.12)(4.7.13) 中间量核对: 0.0
∂s/∂W1 应等于 λ·W1: λW1[0,0]=-0.6763


网络：输入维度 d=4 → 隐藏层 h_dim=5（tanh 激活）→输出 q=3<br>
损失目标函数（带 L2 权重衰减正则）<br>
$J=\underbrace{\frac12\|o-y\|^2}_{L\ 平方损失}+\underbrace{\frac{\lambda}{2}\big(\|W_1\|^2+\|W_2\|^2\big)}_{S\ \text{L2正则惩罚项}}$<br>
关键点：正则项只对权重 W1、W2 的梯度产生贡献，不会往激活值 h、z、o 的误差链上加东西。

.tanh()全称：双曲正切激活函数<br>
公式：<br>
$h=\tanh(z)=\frac{e^{z}-e^{-z}}{e^{z}+e^{-z}}$值域：$\boldsymbol{(-1,\;1)}$<br>
z 很大正数 → tanh (z) → 1<br>
z 很大负数 → tanh (z) → -1<br>
z=0 → tanh(0)=0<br>
.tanh() 导数公式：<br>
$\tanh'(z)=1-\tanh^2(z) = 1-h^2$

W1.norm()<br>
计算 弗罗贝尼乌斯范数 (Frobenius‑norm)，矩阵的 L2 范数。<br>
对于矩阵 W1：<br>
$\|W_1\|_F=\sqrt{\sum_{i}\sum_{k} W_{1,ik}^2}$<br>
把矩阵里面所有元素全部取平方<br>
全部加起来求和<br>
最后开平方根<br>
与 (W1 ** 2).sum() 等价

## 6.d2l 22.4：f=(u+v)² 路径求和 + 反向分解

In [24]:
# ============ d2l 22.4.4 + 22.4.5 例子 ============
# f=(u+v)², u=(a+b)², v=(a-b)², a=(w+x+y+z)², b=(w+x-y-z)²
# 验证：① 多元链式法则=路径求和 ② 反向分解 vs autograd ③ 梯度全 -4096
w, x, y, z = torch.tensor(-1.0), torch.tensor(0.0), torch.tensor(-2.0), torch.tensor(1.0)
w.requires_grad_(True); x.requires_grad_(True); y.requires_grad_(True); z.requires_grad_(True)
a = (w + x + y + z) ** 2
b = (w + x - y - z) ** 2
u = (a + b) ** 2
v = (a - b) ** 2
f = (u + v) ** 2
print("f =", f.item())      # 应为 1024
f.backward()
print("autograd:", w.grad.item(), x.grad.item(), y.grad.item(), z.grad.item())  # 应全 -4096
# --- 反向分解（22.4.5 第二个代码块的逻辑）：先共享中间梯度，再组合 ---
with torch.no_grad():
    df_du = 2 * (u + v);  df_dv = 2 * (u + v)          # ∂f/∂u, ∂f/∂v
    du_da = 2 * (a + b);  du_db = 2 * (a + b)          # ∂u/∂a, ∂u/∂b
    dv_da = 2 * (a - b);  dv_db = -2 * (a - b)         # ∂v/∂a, ∂v/∂b
    df_da = df_du * du_da + df_dv * dv_da              # (22.4.18) 两条路径求和！
    df_db = df_du * du_db + df_dv * dv_db
    da_dv_all = 2 * (w + x + y + z)                    # ∂a/∂w = ∂a/∂x = ∂a/∂y = ∂a/∂z
    db_dv = {"w": 2*(w+x-y-z), "x": 2*(w+x-y-z),       # 注意 b 里 y,z 带负号
             "y": -2*(w+x-y-z), "z": -2*(w+x-y-z)}
for name, v in [("w", w), ("x", x), ("y", y), ("z", z)]:
    manual = df_da * da_dv_all + df_db * db_dv[name]   # 每条变量走两条路径
    print(f"∂f/∂{name}: 手推={manual.item():+.0f}  autograd={v.grad.item():+.0f}")

f = 1024.0
autograd: -4096.0 -4096.0 -4096.0 -4096.0
∂f/∂w: 手推=-4096  autograd=-4096
∂f/∂x: 手推=-4096  autograd=-4096
∂f/∂y: 手推=-4096  autograd=-4096
∂f/∂z: 手推=-4096  autograd=-4096


df_da 那行就是 (22.4.18)：变量 a 通过 u、v 两条路径影响 f，两条路径的贡献相加——多元链式法则的全部秘密
反向分解只算了 4 个共享中间梯度（df_du、df_dv、df_da、df_db）就组合出全部 4 个变量的梯度；而"正向展开"版要对每个变量单独展开一长串、且大量重复（22.4.5 第一个代码块）——你可以把第一个代码块自己补出来对比一下
f=1024、梯度全 −4096：跑出来是这个数，则完成验证

# 四、测试小结

题 1（链式法则展开）
单隐藏层 MLP：输入 $\mathbf{x}\in\mathbb{R}^2$，隐藏层 2 个神经元、激活 tanh，输出层 1 个神经元（无激活），损失 $L=\frac{1}{2}(\hat{y}-y)^2$。令 $\mathbf{z}^{(1)}=\mathbf{W}^{(1)}\mathbf{x}+\mathbf{b}^{(1)}$，$\mathbf{h}=\tanh(\mathbf{z}^{(1)})$，$\hat{y}=\mathbf{w}^{(2)\top}\mathbf{h}+b^{(2)}$。不看任何资料，把 $\partial L/\partial \mathbf{W}^{(1)}$ 沿链式法则逐步展开到能用 $\mathbf{x},\mathbf{h},\hat{y},y,\mathbf{w}^{(2)}$ 显式表达，并标明推导顺序。<br>

$\partial L/\partial \mathbf{W}^{(1)}$ =  $\partial L/\partial \hat{y}$ *  $\partial \hat{y}/\partial h$ * $\partial h/\partial  * \mathbf{z}^{(1)}$ * $\partial \mathbf{z}^{(1)}/\partial \mathbf{W}^{(1)}$ = x * (1 - h^2) * $\mathbf{W}^{(2)}$ * ($\hat{y}$ - y)<br>

在实际代码运行中要注意维度的问题，这里只是数学上的运算
$\frac{\partial L}{\partial \mathbf{W}^{(1)}} = \underbrace{(\hat{y}-y)}_{\text{标量，数乘}} \cdot \Big(\underbrace{\mathbf{w}^{(2)} \odot (1-\mathbf{h}^2)}_{\text{两个 } \mathbb{R}^2 \text{ 向量，逐元素}}\Big) \otimes \mathbf{x}^\top$<br>
⊙ 只在 w^(2) 和 (1−h²) 之间（同形状向量逐元素）<br>
⊗ 只在 ∂L/∂z 和 xᵀ 之间（向量组合出 2×2 矩阵）<br>
(ŷ−y) 是标量，直接乘整个结果

判别口诀：同形状 → ⊙；向量变矩阵 → ⊗；标量 → 直接乘。

题 2（计算顺序）<br>
为什么反向传播必须从输出层往输入层算？如果对每个参数独立展开链式法则（朴素逐个求导），计算复杂度大致是什么量级？一句话说出核心原因。<br>
反向传播复用各层中间结果，避免重复计算；朴素逐个求导每个参数都要跑一遍完整前向，复杂度为 O (参数数量 × 网络深度)，远高于反向传播 O (参数量)，网络越复杂，层数越深那么计算量则会更大；差距就在中间梯度是否共享，反向传播中间梯度共享、每个参数只花 O(1) 组合<br>

题 3（可微性）<br>
激活函数换成阶跃函数 $\sigma(z)=\mathbb{1}[z>0]$，反向传播在哪一步失效？ReLU 在 $z=0$ 不可导，为什么实际训练没问题？<br>
a) z 是连续随机变量，恰好等于 0 的概率为 0；b) 就算踩到，数学上用次梯度（subgradient，0 处可取 [0,1] 中任意值），PyTorch 约定取 0——训练依然有效。

题 4（梯度消失量级估算）<br>
50 层网络，每层 $|w|\approx1$，sigmoid 激活（导数最大值 0.25）。估算 $\partial L/\partial \mathbf{W}^{(1)}$ 相对 $\partial L/\partial \mathbf{W}^{(50)}$ 缩小多少倍（幂表示）？引出什么训练问题？<br>
0.25^49, 因为总共50层，那么第一层要经过后面49层的激活;那么靠近输入层的权重几乎不更新 → 整个网络只有靠近输出层那几层在学 → 深层网络训不动。这就是为什么要有 ReLU、Xavier/He 初始化、残差连接、BN。

题 5（数值梯度）
为什么验证用中心差分而非前向差分？两者的截断误差分别是 $O(h)$ 还是 $O(h^2)$？h 太大/太小分别引入什么误差？结合浮点精度（~1e-16）说明合理 h 的量级。<br>
$f(x+h) = f + hf' + \frac{h^2}{2}f'' + \frac{h^3}{6}f''' + \cdots$<br>
前向差分：$\frac{f(x+h)-f(x)}{h} = f' + \underbrace{\frac{h}{2}f''}_{O(h)} + \cdots$ → 截断误差 O(h)<br>
中心差分：$\frac{f(x+h)-f(x-h)}{2h} = f' + \underbrace{\frac{h^2}{6}f'''}_{O(h^2)} + \cdots$ → 截断误差 O(h²)（奇次项相互抵消）<br>
中心差分更精确，因为误差是 O(h²) 而不是 O(h)

h 太大 → 截断误差主导（O(h²) 项变大）<br>
h 太小 → 舍入误差主导：浮点精度 ~1e-16，分子 (f(x+h)−f(x−h)) 是两个 ~h 量级的小数相减，绝对误差 ~1e-16，除以 h 后误差 ~1e-16/h → h=1e-8 时误差 ~1e-8，h=1e-16 时整个分子直接被吃掉<br>
最优 h：截断误差 O(h²) 和舍入误差 eps/h 平衡 → $h \sim \epsilon^{1/3} \approx 10^{-5}$ 量级，所以验证代码取 h=1e-4 是安全的——这就是 Cell 0 里那个数字的来历<br>